#### Deep Ensemble -- N=3 unabhaengig trainierte Qwen-LoRA-Modelle

Kein Bayes-Sampling, keine Posterior-Approximation -- nur 3 unabhaengige deterministische Trainingslaeufe (Seeds 1, 2, 3), kombiniert per Bayesian Model Averaging (gleicher Mechanismus wie bei MILE/MFVI/Laplace, nur dass die "Samples" hier komplette, separat trainierte Modelle sind). Alle drei Checkpoints sind bereits vortrainiert (`01_train_ensemble_member.ipynb`) und liegen unter `~/Masterarbeit/checkpoints/qwen_lora_agnews_seed{1,2,3}/` -- dieses Notebook trainiert nichts mehr, es laedt nur noch und wertet aus.


In [1]:
from pathlib import Path

# ============================================================
# CONFIG
# DEEP ENSEMBLE (N=3, alle Mitglieder bereits vortrainiert)
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"

BASELINE_SCRIPT = (
    QWEN_REPO / "experiments" / "ag_news_qwen_lora" / "evaluate_saved_baseline.py"
)

RESULT_DIR = MASTER_DIR / "method_results" / "deep_ensemble_qwen_agnews"

# ------------------------------------------------------------
# Ensemble-Mitglieder -- alle drei schon trainiert und gespeichert
# unter ~/Masterarbeit/checkpoints/qwen_lora_agnews_seed{X}/
# (Seed 2 aus dem urspruenglichen Baseline-Training, Seed 1 und 3
# aus 01_train_ensemble_member.ipynb) -- kein Training mehr hier.
# ------------------------------------------------------------

ALL_SEEDS = [1, 2, 3]
N_ENSEMBLE_MEMBERS = len(ALL_SEEDS)

CHECKPOINTS_ROOT = Path.home() / "Masterarbeit" / "checkpoints"

# Nur zur Dokumentation im Summary -- kein Training mehr in diesem Notebook
BATCH_SIZE = 4
NUM_STEPS = 500

# ------------------------------------------------------------
# Posterior data (nur fuer den optionalen Dev-Subset-Eval-Modus,
# Deep Ensemble "lernt" nichts daraus)
# ------------------------------------------------------------

N_PER_CLASS = 512
SEQ_LEN = 32
POSTERIOR_KEY_SEED = 2027

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

EVAL_BATCH_SIZE = 64
EVAL_ON_TEST_SET = True   # direkt auf dem echten Testset (7600 Beispiele)

RUN_NAME = f"deep_ensemble_n{N_ENSEMBLE_MEMBERS}_seq{SEQ_LEN}"

print("======================================")
print("DEEP ENSEMBLE CONFIG")
print("======================================")
print("Run name:          ", RUN_NAME)
print("Seeds:             ", ALL_SEEDS)
print("Total members:     ", N_ENSEMBLE_MEMBERS)
print("======================================")


DEEP ENSEMBLE CONFIG
Run name:           deep_ensemble_n3_seq32
Seeds:              [1, 2, 3]
Total members:      3


In [2]:
# ============================================================
# IMPORTS AND ENVIRONMENT
# ============================================================

import os
import sys
import gc
import copy
import json
import runpy
import subprocess

import numpy as np
import jax
import jax.numpy as jnp

from jax import random


RESULT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(QWEN_REPO)

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

print("\nPaths:")
print("Qwen repo:", QWEN_REPO.exists())
print("Baseline script:", BASELINE_SCRIPT.exists())
print("Result directory:", RESULT_DIR)

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()


Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3
Devices: [CudaDevice(id=0)]

Paths:
Qwen repo: True
Baseline script: True
Result directory: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/deep_ensemble_qwen_agnews


In [3]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


### Alle Mitglieder laden (parametrisierte Kopien von `evaluate_saved_baseline.py`)

Alle drei Seeds werden identisch behandelt -- kein Unterschied mehr zwischen "vorhandenem" und "neuem" Mitglied, da alle Checkpoints bereits existieren. Nutzt exakt denselben, schon bewaehrten Lade-/Eval-Code wie `evaluate_saved_baseline.py`, nur mit ausgetauschtem `CHECKPOINT_PATH`/`SEED`.


In [4]:
# ============================================================
# LOAD ALL MEMBERS (parametrisierte Kopien von evaluate_saved_baseline.py)
# ============================================================

_eval_script_template = BASELINE_SCRIPT.read_text(encoding="utf-8")

_old_checkpoint_line = next(
    line for line in _eval_script_template.splitlines()
    if line.strip().startswith("CHECKPOINT_PATH")
)
_old_seed_line = next(
    line for line in _eval_script_template.splitlines()
    if line.strip().startswith("SEED")
)


def load_ensemble_member(seed):
    seed_dir = CHECKPOINTS_ROOT / f"qwen_lora_agnews_seed{seed}"
    checkpoint_candidates = list(seed_dir.glob("*_pretrained_params.npy"))
    assert len(checkpoint_candidates) == 1, (
        f"Erwarte genau 1 Checkpoint fuer Seed {seed} in {seed_dir}, "
        f"gefunden: {len(checkpoint_candidates)}."
    )
    checkpoint_path = checkpoint_candidates[0]

    script_text = _eval_script_template.replace(
        _old_checkpoint_line, f'CHECKPOINT_PATH = r"{checkpoint_path}"',
    ).replace(
        _old_seed_line, f"SEED = {seed}",
    )

    tmp_script_path = (
        QWEN_REPO / "experiments" / "ag_news_qwen_lora" / f"_tmp_eval_seed{seed}.py"
    )
    tmp_script_path.write_text(script_text, encoding="utf-8")
    try:
        member_objects = runpy.run_path(str(tmp_script_path))
    finally:
        tmp_script_path.unlink(missing_ok=True)

    return member_objects


ensemble_params = {}
ensemble_test_metrics = {}
env = None
data = None
rng_key = None

for seed in ALL_SEEDS:
    print(f"Loading member seed={seed}...")
    member_objects = load_ensemble_member(seed)

    ensemble_params[seed] = member_objects["params"]
    ensemble_test_metrics[seed] = member_objects["test_metrics"]

    if env is None:
        # env/data sind fuer alle Mitglieder identisch (gleiches
        # eingefrorenes Backbone-Modell, gleiche Datenpipeline) --
        # einmal vom ersten geladenen Mitglied uebernehmen.
        env = member_objects["env"]
        data = member_objects["data"]
        rng_key = member_objects["rng_key"]

    print(f"  test_acc={float(ensemble_test_metrics[seed]['test_acc']):.4f}")

print()
print("Alle", len(ensemble_params), "Mitglieder geladen:", sorted(ensemble_params.keys()))


Loading member seed=1...
Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed1/iiljya0g_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.247482
val_acc: 0.912250
val_ece: 0.008029
val_brier: 0.131916
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.252232
test_acc: 0.914474
test_ece: 0.007562
test_brier: 0.132536
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
  test_acc=0.9145
Loading member seed=2...


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps
Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed2/955nvvay_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246214
val_acc: 0.913667
val_ece: 0.009068
val_brier: 0.131597
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251122
test_acc: 0.914474
test_ece: 0.007036
test_brier: 0.131578
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
  test_acc=0.9145
Loading member seed=3...


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]
Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps
Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: /dss/dsshome1/00/ra58vit2/Masterarbeit/checkpoints/qwen_lora_agnews_seed3/7yx76awo_pretrained_params.npy
Saved object shape: (1,)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (12000, 4)
post_predict_logits shape: (7600, 4)
post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.244706
val_acc: 0.916167
val_ece: 0.006206
val_brier: 0.128247
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.258199
test_acc: 0.911184
test_ece: 0.009772
test_brier: 0.134676
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
  test_acc=0.9112

Alle 3 Mitglieder geladen: [1, 2, 3]


In [5]:
# ============================================================
# VECTORIZE ALL MEMBERS
# ============================================================

member_thetas = {}
rebuild_full_params = None

for seed in ALL_SEEDS:
    theta_i, all_leaves_i, lora_indices_i, full_tree_definition_i, unravel_lora_i = (
        qpu.vectorize_lora_params(ensemble_params[seed]["params"])
    )

    member_thetas[seed] = np.asarray(jax.device_get(theta_i))

    if rebuild_full_params is None:
        # Architektur ist bei allen Mitgliedern identisch -- die Struktur
        # (all_leaves/lora_indices/tree_definition/unravel_lora) von Mitglied 1
        # funktioniert fuer alle anderen Mitglieder genauso.
        rebuild_full_params = qpu.build_rebuild_fn(
            all_leaves_i, lora_indices_i, full_tree_definition_i, unravel_lora_i,
        )

sample_thetas = np.stack([member_thetas[s] for s in ALL_SEEDS])

print("sample_thetas shape:", sample_thetas.shape)
print("All finite:", np.isfinite(sample_thetas).all())


LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True
LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True
sample_thetas shape: (3, 540672)
All finite: True


In [6]:
# ============================================================
# EVALUATION DATA (Posterior-Subset oder echtes Testset)
# ============================================================

(
    x_posterior,
    y_posterior,
    posterior_example_keys,
    N_POSTERIOR_EXAMPLES,
    subset_indices_host,
) = qpu.load_posterior_subset(data, N_PER_CLASS, SEQ_LEN, POSTERIOR_KEY_SEED)

if EVAL_ON_TEST_SET:
    x_eval, y_eval = data.get("test")
    print("Evaluating on the full AG News TEST set.")
else:
    x_eval, y_eval = x_posterior, y_posterior
    print("Evaluating on the posterior subset (dev/stability check).")

print("Evaluation examples:", int(y_eval.shape[0]))


Subset indices: [   2   10   13 ... 1902 1907 1910]
Labels: [0 0 0 ... 3 3 3]
Number of examples: 2048
input_ids: (2048, 32)
attention_mask: (2048, 32)
Evaluating on the full AG News TEST set.
Evaluation examples: 7600


In [7]:
# ============================================================
# POSTERIOR PREDICTIVE PROBABILITIES
# ============================================================

sample_probabilities, mean_probabilities, rng_key = (
    qpu.compute_posterior_predictive_probabilities(
        env=env,
        rebuild_full_params=rebuild_full_params,
        sample_thetas=sample_thetas,
        x_eval=x_eval,
        y_eval=y_eval,
        rng_key=rng_key,
        eval_batch_size=EVAL_BATCH_SIZE,
    )
)


Evaluation examples: 7600
Evaluation batch size: 64
Posterior samples: 3
Evaluating posterior sample 1/3
Evaluating posterior sample 2/3
Evaluating posterior sample 3/3

Evaluation completed
Probability tensor shape: (3, 7600, 4)
Expected shape: (3, 7600, 4)
All probabilities finite: True


In [8]:
# ============================================================
# METRICS
# ============================================================

metrics = qpu.compute_predictive_metrics(
    sample_probabilities=sample_probabilities,
    mean_probabilities=mean_probabilities,
    y_true=y_eval,
)


Accuracy: 0.9149999618530273
LPPD: -0.24857616424560547
Posterior predictive NLL: 0.24857616424560547
Brier Score: 0.13108091056346893
Mean predictive entropy: 0.24176084995269775
Mean expected entropy: 0.23816192150115967
Mean mutual information: 0.003598943818360567


In [9]:
# ============================================================
# EXPECTED CALIBRATION ERROR
# ============================================================

ece = qpu.multiclass_ece(mean_probabilities, y_eval, n_bins=15)
print("ECE:", float(ece))


ECE: 0.005560586694628


In [10]:
# ============================================================
# SAVE RUN
# ============================================================

summary, metadata_arrays = qpu.summarize_metrics(metrics)

summary["ece"] = float(ece)

individual_member_test_acc = {
    str(seed): float(ensemble_test_metrics[seed]["test_acc"])
    for seed in ALL_SEEDS
}

summary.update({
    "run_name": RUN_NAME,
    "method": "Deep Ensemble",
    "dataset": "AG News",
    "model": "Qwen2.5-0.5B",

    "n_ensemble_members": N_ENSEMBLE_MEMBERS,
    "seeds": ALL_SEEDS,
    "batch_size": BATCH_SIZE,
    "num_steps_per_member": NUM_STEPS,

    "sequence_length": SEQ_LEN,
    "eval_on_test_set": EVAL_ON_TEST_SET,
    "n_eval_examples": int(y_eval.shape[0]),

    "individual_member_test_accuracy": individual_member_test_acc,

    "n_parameter_dimensions": int(sample_thetas.shape[-1]),
    "all_members_finite": bool(np.isfinite(sample_thetas).all()),
})

metadata_arrays.update({
    "labels": np.asarray(jax.device_get(y_eval)),
    "seeds": np.asarray(ALL_SEEDS),
})

paths = qpu.save_method_run(
    result_dir=RESULT_DIR,
    run_name=RUN_NAME,
    sample_positions=sample_thetas,
    sample_probabilities=sample_probabilities,
    metadata_arrays=metadata_arrays,
    summary=summary,
)

print()
print("Results:")
print("Accuracy:           ", summary["accuracy"])
print("LPPD:               ", summary["lppd"])
print("NLL:                ", summary["posterior_predictive_nll"])
print("Brier Score:        ", summary["brier_score"])
print("ECE:                ", summary["ece"])
print("Mean pred. entropy: ", summary["mean_predictive_entropy"])
print("Mean exp. entropy:  ", summary["mean_expected_entropy"])
print("Mean MI:            ", summary["mean_mutual_information"])
print()
print("Individual member test accuracies:", individual_member_test_acc)



RUN SAVED
Run: deep_ensemble_n3_seq32
Samples:       /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/deep_ensemble_qwen_agnews/deep_ensemble_n3_seq32_samples.npy
Probabilities: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/deep_ensemble_qwen_agnews/deep_ensemble_n3_seq32_probabilities.npy
Metadata:      /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/deep_ensemble_qwen_agnews/deep_ensemble_n3_seq32_metadata.npz
Summary:       /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/deep_ensemble_qwen_agnews/deep_ensemble_n3_seq32_summary.json

Results:
Accuracy:            0.9149999618530273
LPPD:                -0.24857616424560547
NLL:                 0.24857616424560547
Brier Score:         0.13108091056346893
ECE:                 0.005560586694628
Mean pred. entropy:  0.24176086485385895
Mean exp. entropy:   0.23816190659999847
Mean MI:             0.0035989440511912107

Individual member test accuracies: {'1': 0.9144736528396606, '2': 0.9144736528396606, '3': 0.9